# Lab 10: Compressing, detecting, and generating

**DSAN-6600 Deep Learning · Fall 2026**

**Due:** Wednesday Nov 11, 11:59 PM ET · **Target time:** about 75 minutes

---

## What this lab is for

An autoencoder is trained to rebuild its own input through a bottleneck. That
one idea does two very different jobs, and you will build both.

1. **Anomaly detection.** Train on normal data only, then score anything
   unfamiliar by how badly it rebuilds. No labels for the anomalies, which is
   the situation you are actually in.
2. **Generation.** A plain autoencoder's latent space has holes, so you cannot
   draw from it. A VAE closes the holes, and then you can.

Five steps:

1. **The bottleneck.** Build the encoder and decoder and train on MNIST.
2. **The score.** Turn reconstruction error into an anomaly score and pick a
   threshold without ever looking at an anomaly.
3. **The VAE.** Add the two outputs and the reparameterization trick.
4. **The loss.** Reconstruction plus KL, and what happens when you drop one.
5. **Sampling.** Decode points nobody encoded.

## How this lab is graded

- **The notebook is graded for completion, not for correctness.** Submit it,
  with all cells run and outputs visible, and you get the points.
- **Everything is seeded, so your numbers should match a classmate's.** A
  mismatch is a bug, or a cell run out of order. Run All from the top.
- **The understanding is assessed on Quiz 11**, the Thursday after the due date.
- **Solutions are posted after the deadline.**

## Using AI on this lab

Allowed and expected, with disclosure; there is a cell for that at the end.
An agent will write the KL term in one shot. It is much less likely to tell
you why the threshold is set from the training errors and not the test ones,
and Part 6 asks you to.

## Step 0: Setup

::: {.callout-note}
## No GPU required
Both models are small and train for 8 epochs on 60,000 small images. That is
under a minute on a GPU and a few minutes on a laptop CPU. A GPU runtime does
no harm.
:::

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

SEED = 6600
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA = os.environ.get("LAB10_DATA", "./data")
EPOCHS = 8
BATCH = 256

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


def load(which, train):
    """Flattened 0-1 images, one row per image."""
    cls = datasets.MNIST if which == "MNIST" else datasets.FashionMNIST
    ds = cls(DATA, train=train, download=True, transform=transforms.ToTensor())
    return ds.data.float().div(255.0).reshape(len(ds), -1), ds.targets.clone()


x_train, y_train = load("MNIST", True)
x_test, y_test = load("MNIST", False)
x_anom, _ = load("FashionMNIST", False)

print(f"torch {torch.__version__}  device {DEVICE}")
print(f"normal:  {tuple(x_train.shape)} train, {tuple(x_test.shape)} test")
print(f"anomalies: {tuple(x_anom.shape)} (Fashion-MNIST, never trained on)")

---

# Part 1 · The bottleneck

Each image is 784 numbers. The encoder squeezes that down to `latent`, and the
decoder has to rebuild all 784 from only those.

## 1a · Build the two halves

Fill in the encoder and the decoder.

- **Encoder**: `784 -> 256 -> latent`, with `nn.ReLU()` after the first layer
  only. The code itself is not activated.
- **Decoder**: `latent -> 256 -> 784`, with `nn.ReLU()` after the first layer
  and `nn.Sigmoid()` at the very end, because the pixels live in 0 to 1.

In [ ]:
class AE(nn.Module):
    def __init__(self, latent=16):
        super().__init__()
        # TODO: build self.enc and self.dec as nn.Sequential stacks.
        #   enc: Linear(784, 256) -> ReLU -> Linear(256, latent)
        #   dec: Linear(latent, 256) -> ReLU -> Linear(256, 784) -> Sigmoid
        raise NotImplementedError("AE.__init__")

    def forward(self, x):
        z = self.enc(x)
        return self.dec(z), z


torch.manual_seed(SEED)
probe = AE(latent=16)
out, z = probe(x_test[:4])
print(f"in {tuple(x_test[:4].shape)} -> code {tuple(z.shape)} -> out {tuple(out.shape)}")
print(f"parameters: {sum(p.numel() for p in probe.parameters()):,}")
print(f"compression: 784 numbers down to {z.shape[1]}")

## 1b · Train it on MNIST only

Nothing to fill in. Note what the target is: the input itself.

In [ ]:
def train_ae(model, x, epochs=EPOCHS):
    torch.manual_seed(SEED)
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(TensorDataset(x), batch_size=BATCH, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))
    for ep in range(epochs):
        total = 0.0
        for (batch,) in loader:
            batch = batch.to(DEVICE)
            rebuilt, _ = model(batch)
            loss = F.mse_loss(rebuilt, batch)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(batch)
        print(f"  epoch {ep+1:>2}/{epochs}  mse {total/len(x):.5f}")
    return model.eval()


torch.manual_seed(SEED)
ae = train_ae(AE(latent=16), x_train)

In [ ]:
@torch.no_grad()
def rebuild(model, x, chunk=2048):
    out = []
    for i in range(0, len(x), chunk):
        r, _ = model(x[i:i + chunk].to(DEVICE))
        out.append(r.cpu())
    return torch.cat(out)


shown = rebuild(ae, x_test[:8])
fig, axes = plt.subplots(2, 8, figsize=(11, 3))
for j in range(8):
    axes[0, j].imshow(x_test[j].reshape(28, 28), cmap="Greys_r")
    axes[1, j].imshow(shown[j].reshape(28, 28), cmap="Greys_r")
    for r in (0, 1):
        axes[r, j].set_xticks([]); axes[r, j].set_yticks([]); axes[r, j].grid(False)
axes[0, 0].set_ylabel("real", rotation=0, ha="right", va="center")
axes[1, 0].set_ylabel("rebuilt", rotation=0, ha="right", va="center")
fig.suptitle("Rebuilt from 16 numbers")
fig.tight_layout()

---

# Part 2 · The anomaly score

The model has only ever seen handwritten digits. Fashion-MNIST is the same
shape and size, and nothing like it.

## 2a · Error per image

Return one number per image: the **mean squared difference** between the
original and its reconstruction, averaged over the 784 pixels.

In [ ]:
def recon_error(model, x, chunk=2048):
    """One error per image. Returns a 1D tensor of length len(x)."""
    rebuilt = rebuild(model, x, chunk)
    # TODO: squared difference, averaged over pixels (dim=1), one value per row.
    raise NotImplementedError("recon_error")


err_train = recon_error(ae, x_train)
err_test = recon_error(ae, x_test)
err_anom = recon_error(ae, x_anom)
print(f"MNIST train  mean error {err_train.mean():.4f}")
print(f"MNIST test   mean error {err_test.mean():.4f}")
print(f"Fashion      mean error {err_anom.mean():.4f}")

## 2b · The threshold

Set the cut at the **99th percentile of the training errors**. Use
`torch.quantile`.

Read that again: the training errors. Not the test errors, and certainly not
the anomalies. At deployment you have normal data and nothing else, so the
threshold has to come from normal data alone.

In [ ]:
def pick_threshold(err_normal, keep=0.99):
    """The value below which `keep` of the normal errors fall."""
    # TODO: one call to torch.quantile.
    raise NotImplementedError("pick_threshold")


threshold = pick_threshold(err_train)
caught = 100.0 * (err_anom > threshold).float().mean()
false_alarm = 100.0 * (err_test > threshold).float().mean()

print(f"threshold          {threshold:.4f}")
print(f"anomalies caught   {caught:.1f}%")
print(f"false alarms       {false_alarm:.1f}%")

In [ ]:
bins = np.linspace(0, float(torch.quantile(err_anom, 0.995)), 80)
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.hist(err_test.numpy(), bins=bins, color="#2980b9", alpha=0.75,
        label="MNIST, the kind it trained on")
ax.hist(err_anom.numpy(), bins=bins, color="#a8281f", alpha=0.6,
        label="Fashion-MNIST, never seen")
ax.axvline(float(threshold), color="#2c3e50", lw=2.5, ls="--")
ax.set_xlabel("reconstruction error")
ax.set_ylabel("images")
ax.set_title(f"{caught:.0f}% of anomalies past the threshold")
ax.legend()
fig.tight_layout()

## 2c · Look at what it flagged

Nothing to fill in. The worst-rebuilt MNIST digits are the false alarms.

In [ ]:
worst = torch.argsort(err_test, descending=True)[:8]
shown = rebuild(ae, x_test[worst])
fig, axes = plt.subplots(2, 8, figsize=(11, 3))
for j, idx in enumerate(worst):
    axes[0, j].imshow(x_test[idx].reshape(28, 28), cmap="Greys_r")
    axes[1, j].imshow(shown[j].reshape(28, 28), cmap="Greys_r")
    axes[0, j].set_title(f"{err_test[idx]:.3f}", fontsize=10)
    for r in (0, 1):
        axes[r, j].set_xticks([]); axes[r, j].set_yticks([]); axes[r, j].grid(False)
fig.suptitle("The MNIST digits this model rebuilt worst")
fig.tight_layout()

---

# Part 3 · From autoencoder to VAE

Two changes. The encoder produces **two** vectors instead of one, a mean and a
log-variance. And the code handed to the decoder is **drawn** from that
region rather than read off directly.

## 3a · The reparameterization trick

Given `mu` and `logvar`, return a sample.

You cannot write `torch.normal(mu, sigma)` and backpropagate through it. Write
the draw so the randomness is an input instead:

$$z = \mu + \sigma \odot \epsilon, \qquad
  \sigma = \exp(\tfrac{1}{2}\,\text{logvar}), \qquad
  \epsilon \sim N(0, 1)$$

Use `torch.randn_like`.

In [ ]:
def reparameterize(mu, logvar):
    """Draw z from N(mu, sigma^2) so gradients still reach mu and logvar."""
    # TODO: sigma from logvar, then mu + sigma * noise.
    raise NotImplementedError("reparameterize")


torch.manual_seed(SEED)
mu_demo = torch.zeros(5, 2)
logvar_demo = torch.zeros(5, 2)          # logvar 0 means sigma 1
draw = reparameterize(mu_demo, logvar_demo)
print("mu is all zeros and sigma is all ones, so z should look standard normal:")
print(draw)
print(f"\nrequires_grad flows: {reparameterize(mu_demo.requires_grad_(), logvar_demo).requires_grad}")

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent=2):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(784, 256), nn.ReLU())
        self.to_mu = nn.Linear(256, latent)
        self.to_logvar = nn.Linear(256, latent)
        self.dec = nn.Sequential(
            nn.Linear(latent, 256), nn.ReLU(), nn.Linear(256, 784),
            nn.Sigmoid())

    def encode(self, x):
        h = self.body(x)
        return self.to_mu(h), self.to_logvar(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = reparameterize(mu, logvar)
        return self.dec(z), mu, logvar


print("VAE ready. Note encode() returns two things, not one.")

---

# Part 4 · The loss

Two terms. **Reconstruction** keeps the output close to the input.
**KL** pulls the encoder's output distribution toward a standard normal.

For a diagonal Gaussian against $N(0, I)$ the KL has a closed form, which is
on the formula sheet and below:

$$D_{KL} = -\tfrac{1}{2} \sum_j \big(1 + \text{logvar}_j - \mu_j^2
  - e^{\text{logvar}_j}\big)$$

## 4a · Write the two terms

Return `(reconstruction, kl)`, both summed over pixels or latent units and
then **averaged over the batch**, so the two are on the same scale.

Use `F.binary_cross_entropy(..., reduction="sum")` for the reconstruction,
because the pixels are in 0 to 1.

In [ ]:
def vae_loss(rebuilt, x, mu, logvar):
    """Return (reconstruction, kl), each a scalar averaged over the batch."""
    n = len(x)
    # TODO: recon = binary cross entropy, summed, divided by n
    #       kl    = the closed form above, summed, divided by n
    raise NotImplementedError("vae_loss")


torch.manual_seed(SEED)
demo = VAE(latent=2)
r, m, lv = demo(x_test[:64])
rec, kl = vae_loss(r, x_test[:64], m, lv)
print(f"untrained: reconstruction {rec:.1f}   kl {kl:.4f}")
print("kl starts near zero because an untrained encoder already outputs near 0")

## 4b · Train two of them

`beta` is the weight on the KL term. You will train it at `beta=1`, the
ordinary VAE, and at `beta=0`, which removes the KL term entirely and leaves
an autoencoder with extra noise.

In [ ]:
def train_vae(latent=2, beta=1.0, epochs=EPOCHS):
    torch.manual_seed(SEED)
    model = VAE(latent).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(TensorDataset(x_train), batch_size=BATCH, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))
    for ep in range(epochs):
        tot_r = tot_k = 0.0
        for (batch,) in loader:
            batch = batch.to(DEVICE)
            rebuilt, mu, logvar = model(batch)
            rec, kl = vae_loss(rebuilt, batch, mu, logvar)
            loss = rec + beta * kl
            opt.zero_grad(); loss.backward(); opt.step()
            tot_r += rec.item() * len(batch); tot_k += kl.item() * len(batch)
        print(f"  epoch {ep+1:>2}  recon {tot_r/len(x_train):7.2f}  "
              f"kl {tot_k/len(x_train):6.3f}")
    return model.eval()


print("beta = 1, an ordinary VAE")
vae = train_vae(beta=1.0)
print("\nbeta = 0, the KL term switched off")
vae_nokl = train_vae(beta=0.0)

## 4c · Look at what the KL term did

Nothing to fill in. Both models have a two-unit bottleneck and trained for the
same number of epochs. Watch the axis ranges.

In [ ]:
@torch.no_grad()
def codes(model, x, chunk=4096):
    out = []
    for i in range(0, len(x), chunk):
        mu, _ = model.encode(x[i:i + chunk].to(DEVICE))
        out.append(mu.cpu())
    return torch.cat(out).numpy()


z_kl = codes(vae, x_test)
z_nokl = codes(vae_nokl, x_test)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, z, name in ((axes[0], z_nokl, "beta = 0, no KL term"),
                    (axes[1], z_kl, "beta = 1, ordinary VAE")):
    s = ax.scatter(z[:, 0], z[:, 1], c=y_test.numpy(), cmap="tab10", s=2.5,
                   alpha=0.6)
    ax.set_title(f"{name}\nreaches {np.abs(z).max():.0f}")
    ax.set_xlabel("latent 1"); ax.set_ylabel("latent 2")
fig.colorbar(s, ax=axes, ticks=range(10), fraction=0.025, pad=0.02)

print(f"no KL: codes reach {np.abs(z_nokl).max():.1f}")
print(f"KL:    codes reach {np.abs(z_kl).max():.1f}")

---

# Part 5 · Sampling

The point of all of this. Draw a code from a standard normal, decode it, and
see whether you get a digit.

## 5a · Decode a grid

Build a `n` by `n` grid of points over the square from `-span` to `+span` in
both latent coordinates, decode each one, and return the images.

The row order matters for the picture: the **top** row should be the largest
value of latent 2. Return a tensor of shape `(n*n, 784)` in reading order,
left to right then top to bottom.

In [ ]:
@torch.no_grad()
def decode_grid(model, n=15, span=2.5):
    """Decode an n by n grid of latent points. Returns (n*n, 784)."""
    axis = torch.linspace(-span, span, n)
    # TODO: build the (n*n, 2) tensor of points, then decode it.
    #   Row 0 of the picture is the LARGEST latent 2, so walk axis backwards
    #   for the vertical coordinate. torch.flip or axis.flip(0) helps.
    raise NotImplementedError("decode_grid")


grid = decode_grid(vae)
print(f"decoded {len(grid)} points, each {grid.shape[1]} pixels")

In [ ]:
def show_grid(flat, n=15, title=""):
    canvas = np.zeros((n * 28, n * 28))
    imgs = flat.reshape(-1, 28, 28).numpy()
    for i in range(n):
        for j in range(n):
            canvas[i*28:(i+1)*28, j*28:(j+1)*28] = imgs[i * n + j]
    fig, ax = plt.subplots(figsize=(6.4, 6.4))
    ax.imshow(canvas, cmap="Greys_r")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_title(title)
    fig.tight_layout()


show_grid(grid, title="VAE: decoded points, none of which was a real image")
show_grid(decode_grid(vae_nokl), title="No KL term: the same grid")

## 5b · Draw at random

Nothing to fill in. This is the whole claim: a draw from a standard normal
lands somewhere the decoder understands.

In [ ]:
torch.manual_seed(SEED)
z = torch.randn(16, 2)
with torch.no_grad():
    samples = vae.dec(z.to(DEVICE)).cpu()

fig, axes = plt.subplots(2, 8, figsize=(11, 3))
for j, ax in enumerate(axes.flat):
    ax.imshow(samples[j].reshape(28, 28), cmap="Greys_r")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.suptitle("Sixteen draws from N(0, I), decoded")
fig.tight_layout()

---

# Part 6 · What this lab showed

Six questions. If you can answer them, you are ready for the Lab 10 part of
Quiz 11.

1. **Why the threshold comes from the training errors** and not from the test
   set or the anomalies.
2. **What a false alarm is here**, and why choosing the percentile is a
   business decision rather than a modeling one.
3. **Why you cannot backpropagate through a sample**, and what the
   reparameterization trick changes.
4. **What each of the two loss terms does on its own**, and what the
   `beta = 0` latent space looked like compared with `beta = 1`.
5. **Why a draw from a standard normal decodes to a digit** for one of your
   two models and not the other.
6. **Why the samples look soft**, given the loss you trained with.

---

# Part 7 · AI disclosure

Required if you used any generative AI on this lab. Name the tool, say where
you used it, and say what for. "None" is a perfectly good answer.

*Tool(s):*

*Where:*

*What for:*

---

# Submitting

1. **Restart the kernel and Run All.** Confirm it runs top to bottom.
2. Check that the error histogram, both latent scatters, and both decoded
   grids are visible.
3. Render to **HTML or PDF** with resources embedded.
4. Submit the rendered file on Canvas. Due **Wed Nov 11, 11:59 PM ET**.

The notebook is graded for completion. Part 6 is the study list for the quiz.

::: {.callout-note}
## Run from the top if a number looks off
Every training run reseeds inside its own function, so rerunning one cell is
safe. Skipping Part 1 and jumping to Part 2 is not, because `ae` would not
exist.
:::